# 05 - The Five Agent Tools

###  Initialize All Clients

This cell sets up the three components the agent needs to operate:

- **PostgreSQL** - reconnects to the vector database with all FDA label chunks
- **Embeddings client** - calls the BGE Multilingual Gemma2 model via Scaleway Generative APIs to turn queries into vectors
- **LLM client** - connects to Mistral Small 3.2 on Scaleway Generative APIs for reasoning and classification

All three are passed into the `ToolKit`, which is now ready for the agent to use.

### The Agent's Toolkit - 5 Tools

The `ToolKit` class provides the five capabilities the ReAct agent uses to conduct a clinical analysis:

- **`search_drug_kb`** - Performs a general semantic search across all label chunks when the agent needs broad context.
- **`lookup_interactions`** - Targets the interaction sections for a specific drug, which the agent calls for each med in the list.
- **`lookup_population_warnings`** - Retrieves safety alerts specifically for pregnancy, pediatric, geriatric, or renal/hepatic impairment.
- **`flag_severity`** - Classifies each finding as CRITICAL, MAJOR, MODERATE, or MINOR. Findings from a `boxed_warning` are automatically forced to **CRITICAL**.
- **`summarize_evidence`** - Synthesizes the final structured report, ensuring every claim is backed by a verbatim evidence snippet and a traceable source ID.

In [ ]:
import os
import sys
import subprocess
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

# Connect to database
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)

# Embeddings client (Scaleway Generative APIs, serverless)
from src.embeddings import EmbeddingsClient

embedding_client = EmbeddingsClient(
    client=OpenAI(base_url="https://api.scaleway.ai/v1", api_key=os.environ["SCW_SECRET_KEY"]),
    model="bge-multilingual-gemma2",
    dimensions=768,
)

# LLM client
llm_client = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

# Create toolkit
from src.tools import ToolKit

toolkit = ToolKit(conn=conn, embeddings_client=embedding_client, llm_client=llm_client)

print("Toolkit ready.")

###  Cell 2 - Test Tool 1: `search_drug_kb`

A quick test of the first tool - searching the knowledge base with a broad clinical query: *"bleeding risk"*.

The agent uses this tool when it does not yet know which drug or section to look in.
It returns the most semantically similar chunk from the entire database.

In [ ]:
drug_name = "KETOCONAZOLE"

# Tool 1: search_drug_kb
results = toolkit.search_drug_kb("bleeding risk", top_k=1)

print("search_drug_kb('bleeding risk')")
print("===")
for r in results:
    print(f"  {r['drug_name']} | {r['section_type']} | {r.get('distance', 'N/A')}")
    print(f"  {r['text'][:120]}...")
    print()

###  Cell 3 - Test Tool 2: `lookup_interactions`

This cell tests the second tool using a specific drug: `KETOCONAZOLE`.

**Ketoconazole** is a synthetic antifungal medication.

**Ketoconazole** has numerous clinically significant interactions, primarily acting as a potent **CYP3A4 enzyme inhibitor**. This causes other drugs to accumulate in the bloodstream, sharply increasing the risk of toxicity. It is strictly contraindicated with cisapride, astemizole, terfenadine, and pimozide (due to QT prolongation risk), as well as drugs that reduce stomach acidity.

Unlike the broad search, this tool is **targeted**: it searches only within the `drug_interactions` section for the chosen drug. This is the primary tool the agent uses to gather evidence for each medication in the user's list.

In [ ]:
# Tool 2: lookup_interactions - KETOCONAZOLE
results = toolkit.lookup_interactions(f"{drug_name}")

print(f"lookup_interactions('{drug_name}')")
print("===")
for r in results:
    print(f"  Section: {r['section_type']} | distance: {r.get('distance', 'N/A')}")
    print(f"  {r['text'][:200]}...")
    print()

### Cell 4 - Test Tool 3: `lookup_population_warnings` (Pediatric)

Tests population-specific warnings for **KETOCONAZOLE** in the **pediatric** population.

The tool searches the `pediatric_use` section of the FDA label and returns any safety information relevant to children - such as dosage restrictions, contraindications, or lack of clinical data for this age group.

In [ ]:
# Tool 3: lookup_population_warnings - metformin renal
results = toolkit.lookup_population_warnings(drug_name, "pediatric")

print(f"lookup_population_warnings({drug_name}, 'pediatric')")
print("===")
for r in results:
    print(f"  Section: {r['section_type']}")
    print(f"  {r['text'][:500]}...")
    print()

###  Cell 6 - Test Tools 4 & 5: `flag_severity` and `summarize_evidence`

This cell tests the final two tools using a sample finding about KETOCONAZOLE.

**`flag_severity`** classifies the finding and assigns a severity level - CRITICAL, MAJOR, MODERATE, or MINOR.  
**`summarize_evidence`** then takes the classified findings and produces the final structured report with a claim, source ID, and verbatim evidence snippet for each entry.

> 💡 These two tools are always called last - they turn raw retrieved evidence into a clean, traceable clinical report.

In [ ]:
# Tools 4 & 5: flag_severity and summarize_evidence
sample_findings = [
    {
        "claim": "KETOCONAZOLE + CYP3A4 increases bleeding risk ",
        "source_section_type": "drug_interactions",
        "source_id": "KETOCONAZOLE",
        "evidence_snippet": "Drugs that Increase Bleeding Risk: KETOCONAZOLE, CYP3A4",
    }
]

flagged = toolkit.flag_severity(sample_findings)
print("flag_severity results (should be severity-ordered):")
for f in flagged:
    print(f"  [{f['severity']}] {f['claim']}")

print()

summary = toolkit.summarize_evidence(flagged)
print("summarize_evidence results:")
print(json.dumps(summary, indent=2))

## You should now have

- [x] Tested all five tools individually
- [x] Seen grounded citations with set_id references
- [x] Understood severity classification (boxed_warning = CRITICAL)

**Next:** `06_react_agent.ipynb`